# Lab 2: Data Extraction, Transformation & Missing Value Exploration
**Topic:** Global Health Data Platform & Life Expectancy Calculator  
**Author:** Tudor Stanichevici  
**Date:** 2026-11-04  

This notebook fulfills **Task 2.1** of Lab 2:
1. Calling raw extractors for WHO, World Bank, and Our World in Data.
2. Transforming raw JSON / CSV files into normalized observation DataFrames (`country_iso3`, `indicator_code`, `year`, `sex`, `value`).
3. Inspecting datasets with `.head()`, `.info()`, and `.isna().sum()`.
4. Calculating the share of missing values per indicator and analyzing data completeness across sources.


In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add repository root to python path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from etl.extract_worldbank import download_worldbank_countries, download_worldbank_indicator
from etl.extract_who import download_who_indicator
from etl.extract_owid import download_owid_chart

from etl.transform_worldbank import transform_worldbank_country_file, transform_worldbank_indicator_file
from etl.transform_who import transform_who_file
from etl.transform_owid import transform_owid_chart_file


## 1. Extract Raw Data
We extract raw data from the three primary sources:
- **World Bank API v2**: Sovereign countries metadata and indicators (`SH.XPD.CHEX.GD.ZS`, `NY.GDP.PCAP.CD`, `SP.DYN.IMRT.IN`, `SH.IMM.MEAS`).
- **WHO Global Health Observatory (GHO)**: Life expectancy at birth (`WHOSIS_000001`), adult tobacco use (`M_Est_tob_curr_std`), alcohol consumption (`SA_0000001400`).
- **Our World in Data (OWID)**: Historical life expectancy chart (`life-expectancy`).


In [2]:
# 1. World Bank Extraction
wb_country_path, wb_country_url = download_worldbank_countries()
print(f"World Bank Countries: {wb_country_path.name} from {wb_country_url}")

wb_indicators = ["SH.XPD.CHEX.GD.ZS", "NY.GDP.PCAP.CD", "SP.DYN.IMRT.IN", "SH.IMM.MEAS"]
wb_paths = {}
for code in wb_indicators:
    p, url = download_worldbank_indicator(code, "2000:2023")
    wb_paths[code] = p
    print(f"World Bank {code}: {p.name}")

# 2. WHO Extraction
who_indicators = ["WHOSIS_000001", "M_Est_tob_curr_std", "SA_0000001400"]
who_paths = {}
for code in who_indicators:
    p, url = download_who_indicator(code, 2000)
    who_paths[code] = p
    print(f"WHO {code}: {p.name}")

# 3. OWID Extraction
owid_slug = "life-expectancy"
owid_path, owid_url = download_owid_chart(owid_slug)
print(f"OWID {owid_slug}: {owid_path.name} from {owid_url}")


World Bank Countries: countries.json from https://api.worldbank.org/v2/country


World Bank SH.XPD.CHEX.GD.ZS: SH.XPD.CHEX.GD.ZS_2000-2023.json


World Bank NY.GDP.PCAP.CD: NY.GDP.PCAP.CD_2000-2023.json


World Bank SP.DYN.IMRT.IN: SP.DYN.IMRT.IN_2000-2023.json


World Bank SH.IMM.MEAS: SH.IMM.MEAS_2000-2023.json


WHO API unavailable (503 Server Error: Service Unavailable for url: https://ghoapi.azureedge.net/api/WHOSIS_000001?%24filter=SpatialDimType+eq+%27COUNTRY%27+and+TimeDim+ge+2000); falling back to cached response who_WHOSIS_000001.json


WHO WHOSIS_000001: WHOSIS_000001_from2000.json


WHO API unavailable (503 Server Error: Service Unavailable for url: https://ghoapi.azureedge.net/api/M_Est_tob_curr_std?%24filter=SpatialDimType+eq+%27COUNTRY%27+and+TimeDim+ge+2000); falling back to cached response who_M_Est_tob_curr_std.json


WHO M_Est_tob_curr_std: M_Est_tob_curr_std_from2000.json


WHO API unavailable (503 Server Error: Service Unavailable for url: https://ghoapi.azureedge.net/api/SA_0000001400?%24filter=SpatialDimType+eq+%27COUNTRY%27+and+TimeDim+ge+2000); falling back to cached response who_SA_0000001400.json


WHO SA_0000001400: SA_0000001400_from2000.json
OWID life-expectancy: life-expectancy_full.csv from https://ourworldindata.org/grapher/life-expectancy.csv?v=1&csvType=full&useColumnShortNames=true


## 2. Transform Raw Data to Normalized DataFrames
Next, we transform raw payloads into clean tabular representations.


In [3]:
# Transform World Bank Countries
df_countries = transform_worldbank_country_file(wb_country_path)
print(f"Valid sovereign economies transformed: {len(df_countries)}")
df_countries.head()


Valid sovereign economies transformed: 217


,iso3,iso2,name,region,income_group
0,ABW,AW,Aruba,Latin America & Caribbean,High income
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income
4,AND,AD,Andorra,Europe & Central Asia,High income


In [4]:
# Transform World Bank Indicators
transformed_dfs = {}
for code, path in wb_paths.items():
    transformed_dfs[code] = transform_worldbank_indicator_file(path, code)

# Transform WHO Indicators
for code, path in who_paths.items():
    transformed_dfs[code] = transform_who_file(path, code)

# Transform OWID
transformed_dfs["OWID_LIFE_EXPECTANCY"] = transform_owid_chart_file(owid_path, "OWID_LIFE_EXPECTANCY")


## 3. Dataset Inspections (head, info, isna)
We inspect the structure and null patterns of indicators across each provider.


In [5]:
# 3.1 World Bank: Health Expenditure % GDP
df_chex = transformed_dfs["SH.XPD.CHEX.GD.ZS"]
print("=== SH.XPD.CHEX.GD.ZS head() ===")
print(df_chex.head())
print("\n=== SH.XPD.CHEX.GD.ZS info() ===")
df_chex.info()
print("\n=== SH.XPD.CHEX.GD.ZS isna().sum() ===")
print(df_chex.isna().sum())


=== SH.XPD.CHEX.GD.ZS head() ===
  country_iso3     indicator_code  year   sex     value
0          AFE  SH.XPD.CHEX.GD.ZS  2023  None  5.543797
1          AFE  SH.XPD.CHEX.GD.ZS  2022  None  5.566469
2          AFE  SH.XPD.CHEX.GD.ZS  2021  None  5.946320
3          AFE  SH.XPD.CHEX.GD.ZS  2020  None  5.985494
4          AFE  SH.XPD.CHEX.GD.ZS  2019  None  5.754125

=== SH.XPD.CHEX.GD.ZS info() ===
<class 'pandas.DataFrame'>
RangeIndex: 6384 entries, 0 to 6383
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6384 non-null   str    
 1   indicator_code  6384 non-null   str    
 2   year            6384 non-null   str    
 3   sex             0 non-null      object 
 4   value           5692 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 249.5+ KB

=== SH.XPD.CHEX.GD.ZS isna().sum() ===
country_iso3         0
indicator_code       0
year                 0
sex             

In [6]:
# 3.2 WHO: Life Expectancy at Birth
df_who_le = transformed_dfs["WHOSIS_000001"]
print("=== WHOSIS_000001 head() ===")
print(df_who_le.head())
print("\n=== WHOSIS_000001 info() ===")
df_who_le.info()
print("\n=== WHOSIS_000001 isna().sum() ===")
print(df_who_le.isna().sum())


=== WHOSIS_000001 head() ===
  country_iso3 indicator_code  year       sex  value
0          SOM  WHOSIS_000001  2008   SEX_MLE   48.1
1          BTN  WHOSIS_000001  2002  SEX_BTSX   67.6
2          BHR  WHOSIS_000001  2011  SEX_FMLE   78.7
3          CYP  WHOSIS_000001  2003   SEX_MLE   76.9
4          DNK  WHOSIS_000001  2014   SEX_MLE   78.6

=== WHOSIS_000001 info() ===
<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    13320 non-null  str    
 1   indicator_code  13320 non-null  str    
 2   year            13320 non-null  int64  
 3   sex             13320 non-null  str    
 4   value           13320 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 520.4 KB

=== WHOSIS_000001 isna().sum() ===
country_iso3      0
indicator_code    0
year              0
sex               0
value             0
dtype: int64


In [7]:
# 3.3 OWID: Historical Life Expectancy
df_owid_le = transformed_dfs["OWID_LIFE_EXPECTANCY"]
print("=== OWID Life Expectancy head() ===")
print(df_owid_le.head())
print("\n=== OWID Life Expectancy info() ===")
df_owid_le.info()
print("\n=== OWID Life Expectancy isna().sum() ===")
print(df_owid_le.isna().sum())


=== OWID Life Expectancy head() ===
  country_iso3        indicator_code  year   sex    value
0          AFG  OWID_LIFE_EXPECTANCY  1950  None  28.1563
1          AFG  OWID_LIFE_EXPECTANCY  1951  None  28.5836
2          AFG  OWID_LIFE_EXPECTANCY  1952  None  29.0138
3          AFG  OWID_LIFE_EXPECTANCY  1953  None  29.4521
4          AFG  OWID_LIFE_EXPECTANCY  1954  None  29.6975

=== OWID Life Expectancy info() ===
<class 'pandas.DataFrame'>
RangeIndex: 21565 entries, 0 to 21564
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    20217 non-null  str    
 1   indicator_code  21565 non-null  str    
 2   year            21565 non-null  int64  
 3   sex             0 non-null      object 
 4   value           21565 non-null  float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 842.5+ KB

=== OWID Life Expectancy isna().sum() ===
country_iso3       1348
indicator_code        0
year

## 4. Share of Missing Values per Indicator
We compute the exact share of missing values for all downloaded indicators to evaluate data density and reporting gaps.


In [8]:
missing_stats = []

for indicator_code, df in transformed_dfs.items():
    total_rows = len(df)
    missing_val = int(df["value"].isna().sum())
    missing_pct = (missing_val / total_rows * 100) if total_rows > 0 else 0.0
    missing_iso = int(df["country_iso3"].isna().sum())
    missing_year = int(df["year"].isna().sum())
    
    missing_stats.append({
        "Indicator Code": indicator_code,
        "Total Rows": total_rows,
        "Missing Values": missing_val,
        "Missing Value %": round(missing_pct, 2),
        "Missing Country ISO3": missing_iso,
        "Missing Year": missing_year,
    })

df_summary = pd.DataFrame(missing_stats)
print(df_summary.to_string(index=False))
df_summary


      Indicator Code  Total Rows  Missing Values  Missing Value %  Missing Country ISO3  Missing Year
   SH.XPD.CHEX.GD.ZS        6384             692            10.84                     0             0
      NY.GDP.PCAP.CD        6384             190             2.98                     0             0
      SP.DYN.IMRT.IN        6384             528             8.27                     0             0
         SH.IMM.MEAS        6384             619             9.70                     0             0
       WHOSIS_000001       13320               0             0.00                     0             0
  M_Est_tob_curr_std        5511               0             0.00                     0             0
       SA_0000001400           0               0             0.00                     0             0
OWID_LIFE_EXPECTANCY       21565               0             0.00                  1348             0


,Indicator Code,Total Rows,Missing Values,Missing Value %,Missing Country ISO3,Missing Year
0,SH.XPD.CHEX.GD.ZS,6384,692,10.84,0,0
1,NY.GDP.PCAP.CD,6384,190,2.98,0,0
2,SP.DYN.IMRT.IN,6384,528,8.27,0,0
3,SH.IMM.MEAS,6384,619,9.70,0,0
4,WHOSIS_000001,13320,0,0.00,0,0
5,M_Est_tob_curr_std,5511,0,0.00,0,0
6,SA_0000001400,0,0,0.00,0,0
7,OWID_LIFE_EXPECTANCY,21565,0,0.00,1348,0


### Observations & Findings:
1. **World Bank Regular Grids**:
   - The World Bank API returns a pre-indexed Cartesian product of all economies across all requested years (2000–2023). When an economy did not report data in a given year (e.g. recent health spending in conflict zones or small islands), the API sends `{"value": null}`. Consequently, World Bank indicators naturally display missing values (~10% to 15% depending on indicator reporting velocity).
2. **WHO GHO Sparsity**:
   - The WHO Global Health Observatory API returns records *only* when a measurement, estimate, or survey was actually recorded (`NumericValue` populated). As a result, transformed WHO indicator records show 0% null values in the raw payload; missingness here is structural (missing rows rather than null values).
3. **Our World in Data (OWID)**:
   - The OWID curated long-format CSV omits empty measurements, resulting in 0% nulls in the `value` column. However, it contains sub-national and regional codes (e.g. `OWID_WRL`) where `code` is empty or non-ISO, which will be filtered out by `clean_observations()` in the cleaning step.
